In [1]:
# Importar bibliotecas
# -----------------------------------------------------------------------

# Manejo de data
# -----------------------------------------------------------------------
import pandas as pd
import numpy as np
import re

# Visualizacion
# -----------------------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency, fisher_exact

# Preprocesamiento de datos y modelado de valores faltantes
# -----------------------------------------------------------------------
import word2number
from word2number import w2n
from sklearn.impute import SimpleImputer 
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer 
from sklearn.impute import KNNImputer 
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from pandas.api.types import is_numeric_dtype

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from statsmodels.graphics.mosaicplot import mosaic

# Configuraciones
# -----------------------------------------------------------------------
pd.set_option('display.max_columns', None) 
pd.set_option('display.float_format', '{:.2f}'.format)

# Estilo general
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
from IPython.display import display

import warnings
import matplotlib
matplotlib.rcParams.update({'figure.max_open_warning': 0})
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=UserWarning)

In [2]:
def cargar_archivo(file_path, delimiter=',', index_col=None): 
   """ 
    Carga un archivo CSV en el DataFrame de Pandas.

    Parámetros:
        file_path (str): Ruta al archivo CSV.
        delimiter (str): Delimitador utilizado en el archivo CSV (el valor predeterminado es ',').
        index_col (str o int, opcional): Columna que se usará como etiqueta de fila del DataFrame.

    Retorna:
        pd.DataFrame: DataFrame que contiene los datos CSV.
   """
   try:
        df = pd.read_csv(file_path, delimiter=delimiter, index_col=index_col)
        print('Archivo CSV cargado exitosamente.')
        return pd.read_csv(file_path, delimiter=delimiter, index_col=index_col)
   except FileNotFoundError:
       print(f'Error: Archivo no encontrado en {file_path}. Por favor revise la ruta.')
   except Exception as e:
       print(f'Ha ocurrido un error: {e}')

In [3]:
df = cargar_archivo('adult.csv')
df

Archivo CSV cargado exitosamente.


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,22,Private,310152,Some-college,10,Never-married,Protective-serv,Not-in-family,White,Male,0,0,40,United-States,<=50K
32557,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32558,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32559,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K


In [4]:
# Limpiar categoria, una sola (married) ✔️
# Eliminar nulos (?) ✔️
# unificar nombre de columnas ✔️
# nueva columna de generacion ✔️
# eliminar '-' en valores ✔️

In [5]:
def reemplazar_puntos_columnas(df):
    """
    Reemplaza los puntos ('.') por guiones bajos ('_') en los nombres de las columnas de un DataFrame.
    
    Parámetros:
        df (pd.DataFrame): DataFrame de pandas.
    
    Retorna:
        pd.DataFrame: DataFrame con nombres de columnas modificados.
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError("El parámetro 'df' debe ser un DataFrame de pandas.")
    
    df.columns = [col.replace('.', '_') for col in df.columns]
    
    return df

In [6]:
reemplazar_puntos_columnas(df)
df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education_num   32561 non-null  int64 
 5   marital_status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital_gain    32561 non-null  int64 
 11  capital_loss    32561 non-null  int64 
 12  hours_per_week  32561 non-null  int64 
 13  native_country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


In [8]:
def limpiar_dataframe(df):
    """
    Esta función recibe un DataFrame, reemplaza los valores NaN por '?',
    y elimina cualquier fila que contenga al menos un '?'.
    """
    df_reemplazado = df.fillna('?')
    df = df_reemplazado[~df_reemplazado.isin(['?']).any(axis=1)]
    
    return df

In [9]:
df = limpiar_dataframe(df)

In [10]:
def reemplazar_valores(df):
    """
    Reemplazar todas las apariciones de '-' con espacios en blanco en un DataFrame de Pandas.

    Parámetros:
        df(pd.DataFrame): El DataFrame de entrada.

    Retorna:
        pd.DataFrame: Un nuevo DataFrame con los reemplazos aplicados.
    """   
    return df.replace({'-':' '}, regex=True)

In [11]:
df = reemplazar_valores(df)
df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
1,82,Private,132870,HS grad,9,Widowed,Exec managerial,Not in family,White,Female,0,4356,18,United States,<=50K
3,54,Private,140359,7th 8th,4,Divorced,Machine op inspct,Unmarried,White,Female,0,3900,40,United States,<=50K
4,41,Private,264663,Some college,10,Separated,Prof specialty,Own child,White,Female,0,3900,40,United States,<=50K
5,34,Private,216864,HS grad,9,Divorced,Other service,Unmarried,White,Female,0,3770,45,United States,<=50K
6,38,Private,150601,10th,6,Separated,Adm clerical,Unmarried,White,Male,0,3770,40,United States,<=50K


In [12]:
def mostrar_resumen_categorico(df):
    """ 
    Muestra el recuento de valores de cada columna categórica del marco de datos.

    Parámetros:
        df(pd.DataFrame): El marco de datos que se analizará.

    Retorna:
        Marcos de datos con las métricas de las columnas categóricas.
    """
    summaries = {}
    for col in df.select_dtypes(include='O').columns:
        counts = df[col].value_counts().reset_index()
        summaries[col] = counts
        print(f"\n📊 Categoria por columna: {col}")
        print(counts.to_string(index=False))
        print('—' * 60)
        
    return summaries

In [13]:
mostrar_resumen_categorico(df)


📊 Categoria por columna: workclass
       workclass  count
         Private  22286
Self emp not inc   2499
       Local gov   2067
       State gov   1279
    Self emp inc   1074
     Federal gov    943
     Without pay     14
————————————————————————————————————————————————————————————

📊 Categoria por columna: education
   education  count
     HS grad   9840
Some college   6678
   Bachelors   5044
     Masters   1627
   Assoc voc   1307
        11th   1048
  Assoc acdm   1008
        10th    820
     7th 8th    557
 Prof school    542
         9th    455
        12th    377
   Doctorate    375
     5th 6th    288
     1st 4th    151
   Preschool     45
————————————————————————————————————————————————————————————

📊 Categoria por columna: marital_status
       marital_status  count
   Married civ spouse  14065
        Never married   9726
             Divorced   4214
            Separated    939
              Widowed    827
Married spouse absent    370
    Married AF spouse     21
—

{'workclass':           workclass  count
 0           Private  22286
 1  Self emp not inc   2499
 2         Local gov   2067
 3         State gov   1279
 4      Self emp inc   1074
 5       Federal gov    943
 6       Without pay     14,
 'education':        education  count
 0        HS grad   9840
 1   Some college   6678
 2      Bachelors   5044
 3        Masters   1627
 4      Assoc voc   1307
 5           11th   1048
 6     Assoc acdm   1008
 7           10th    820
 8        7th 8th    557
 9    Prof school    542
 10           9th    455
 11          12th    377
 12     Doctorate    375
 13       5th 6th    288
 14       1st 4th    151
 15     Preschool     45,
 'marital_status':           marital_status  count
 0     Married civ spouse  14065
 1          Never married   9726
 2               Divorced   4214
 3              Separated    939
 4                Widowed    827
 5  Married spouse absent    370
 6      Married AF spouse     21,
 'occupation':            occupation  co

In [14]:
def unificar_valores(df, columna='marital_status'):
    """
    Normaliza los valores de la columna 'marital_status' en el DataFrame.
    Unifica todos los valores que contienen 'married' en uno solo: 'Married'.
    
    Parámetros:
        df (pd.DataFrame): DataFrame de entrada.
        columna (str): Nombre de la columna a unificar.
        
    Retorna:
        pd.DataFrame: DataFrame con valores unificados en la columna especificada.
    """
    mapping = {
        'married civ spouse': 'Married',
        'married spouse absent': 'Married',
        'married af spouse': 'Married'
    }

    df[columna] = df[columna].apply(lambda x: mapping.get(str(x).lower(), x))
    
    return df

In [15]:
df = unificar_valores(df)

In [16]:
def asignar_generacion(df, columna_edad='age', nueva_columna='generacion'):
    """
    Crea una nueva columna con la generación correspondiente según la edad.

    Parámetros:
        df (pd.DataFrame): DataFrame de entrada.
        columna_edad (str): Nombre de la columna que contiene la edad.
        nueva_columna (str): Nombre de la nueva columna a crear.

    Retorna
        pd.DataFrame: DataFrame con la nueva columna de generación.
    """
    def clasificar_generacion(edad):
        if edad >= 77:
            return 'Silent Generation'
        elif 59 <= edad <= 76:
            return 'Baby Boomer'
        elif 43 <= edad <= 58:
            return 'Gen X'
        elif 27 <= edad <= 42:
            return 'Millennial'
        elif 11 <= edad <= 26:
            return 'Gen Z'
        elif edad <= 10:
            return 'Gen Alpha'
        else:
            return 'Desconocido'

    df[nueva_columna] = df[columna_edad].apply(clasificar_generacion)
    
    return df

In [17]:
df = asignar_generacion(df)

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30162 entries, 1 to 32560
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             30162 non-null  int64 
 1   workclass       30162 non-null  object
 2   fnlwgt          30162 non-null  int64 
 3   education       30162 non-null  object
 4   education_num   30162 non-null  int64 
 5   marital_status  30162 non-null  object
 6   occupation      30162 non-null  object
 7   relationship    30162 non-null  object
 8   race            30162 non-null  object
 9   sex             30162 non-null  object
 10  capital_gain    30162 non-null  int64 
 11  capital_loss    30162 non-null  int64 
 12  hours_per_week  30162 non-null  int64 
 13  native_country  30162 non-null  object
 14  income          30162 non-null  object
 15  generacion      30162 non-null  object
dtypes: int64(6), object(10)
memory usage: 3.9+ MB


In [19]:
def guardar_datos_en_csv(df, file_path, include_index=False):
    """
    Saves a Pandas DataFrame to a CSV file.

    Parámetros:
    - dataframe (pd.DataFrame): The DataFrame to save.
    - file_path (str): The path (including filename) where the CSV will be saved.
    - include_index (bool): Whether to include the DataFrame's index in the CSV (default is False).
    """
    try:
        df.to_csv(file_path, index=include_index)
        print(f"DataFrame guardado con exito {file_path}")
    except Exception as e:
        print(f"Ha ocurrido un error al guardar el DataFrame: {e}")

In [20]:
guardar_datos_en_csv(df, 'adult_limpio.csv')

DataFrame guardado con exito adult_limpio.csv
